Full name: Elsa Ingrid Daniela Erkfeldt
Civic registration number: 200309021228
LLM:

In [1]:
import tensorflow as tf
import os
import torch.nn as nn
import numpy as np
import torch

if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

Load the data from the PTB data set

In [2]:
url = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/"
files = ["ptb.train.txt", "ptb.valid.txt", "ptb.test.txt"]

for f in files: 
    tf.keras.utils.get_file(f, url + f)

def load_data(filename):
    path = os.path.join(os.path.expanduser('~'), '.keras/datasets', filename)
    with open(path, 'r', encoding = 'utf-8') as f:
        return f.read().replace('\n', '<eos>')

train_text = load_data('ptb.train.txt')
valid_text = load_data('ptb.valid.txt')
test_text = load_data('ptb.test.txt')


5101618/5101618 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
399782/399782 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
449945/449945 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [32]:
print(train_text[50])

m


In [22]:
#Split the data into tokens and sort them
tokens = train_text.split()
vocab = sorted(list(set(tokens)))

valid_tok = valid_text.split()
#valid_vocab = sorted(list(set(valid_tok)))

#Create mappings to be able to get an index from a word and vice versa
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}

vocab_size = len(vocab)
seq_length = 32

#Create datasets with ser_length words as X and the word after that as y
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(0, len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(tokens, seq_length)
X_valid, y_valid = create_sequences(valid_tok, seq_length)

In [30]:
#Reuse the LSTM model from assignment 1 with some changes
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers, dropout):
        super(LSTMModel, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size) #converts word IDs to vectors. vocab_size = num of distinct tokens. Embed_size = how big each word vector is 
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.linear = nn.Linear(hidden_size, vocab_size)
    def forward(self, x, h): #x:(batch_size, seq_length), h: (h0, c0), h0: zeros in shape (num_layers, batch_size, hidden_size), c0: same
        x = self.embed(x)
        out, (h,c) = self.lstm(x, h)
        out = out.reshape(out.size(0)*out.size(1), out.size(2))
        out = self.linear(out)
        return out, (h, c)

embed_size = 200
hidden_size = 200
num_layers = 2
learning_rate = 0.001

#Train the model on the data set we created
model = LSTMModel(vocab_size, embed_size, hidden_size, num_layers, dropout=0.2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [5]:
"""#Validation
model.eval()

#predict
h0_predict = torch.zeros(num_layers, 1, hidden_size, , device=device)
c0_predict = torch.zeros(num_layers, 1, hidden_size, device = device)
val_per_time_test, _ =model(X_valid[0], (h0_predict, c0_predict))
print(val_per_time_test[-1])
id_for_last = torch.argmax(val_per_time_test[-1])
word_for_last = idx2word[id_for_last]
print(word_for_last)"""

'#Validation\nmodel.eval()\n\n#predict\nh0_predict = torch.zeros(num_layers, 1, hidden_size, , device=device)\nc0_predict = torch.zeros(num_layers, 1, hidden_size, device = device)\nval_per_time_test, _ =model(X_valid[0], (h0_predict, c0_predict))\nprint(val_per_time_test[-1])\nid_for_last = torch.argmax(val_per_time_test[-1])\nword_for_last = idx2word[id_for_last]\nprint(word_for_last)'

In [6]:
"""#A function to calcualte the probability that the model will predict correctly

def predict_next(model, word, sentence, word2idx):


    model.eval()


    h0_predict = torch.zeros(num_layers, 1, hidden_size, , device=device)
    c0_predict = torch.zeros(num_layers, 1, hidden_size, device = device)
    val_per_time_test, _ = model(sentence, (h0_predict, c0_predict))
    print(val_per_time_test[-1])
    probs_next_word = val_per_time_test[-1]

    correct_next_word = word
    correct_next_word_id = word2idx[correct_next_word]
    prob_correct = probs_next_word[correct_next_word_id]


    id_for_last = torch.argmax(probs_next_word)
    word_for_last = idx2word[id_for_last]
    print(word_for_last)

    return prob_correct




predict_next(model, y_train[0], X_train[0], word2idx)

#Function to calculate perplexity
def calculate_perplexity(words):
    # PP = (P (tn | t1,..., t_n-1) ... P(t_3|t_1, t_2)P(t_2|t_1)P(t_1))^{-1/n}
    n = len(words)
    p_inside = 1
    for i in range(len(words), 0, -1):

        p_inside *= predict_next(model, words[i], words[0:i],word2idx)
    PP = (p_inside)**(-1/n)"""

'#A function to calcualte the probability that the model will predict correctly\n\ndef predict_next(model, word, sentence, word2idx):\n\n\n    model.eval()\n\n\n    h0_predict = torch.zeros(num_layers, 1, hidden_size, , device=device)\n    c0_predict = torch.zeros(num_layers, 1, hidden_size, device = device)\n    val_per_time_test, _ = model(sentence, (h0_predict, c0_predict))\n    print(val_per_time_test[-1])\n    probs_next_word = val_per_time_test[-1]\n\n    correct_next_word = word\n    correct_next_word_id = word2idx[correct_next_word]\n    prob_correct = probs_next_word[correct_next_word_id]\n\n\n    id_for_last = torch.argmax(probs_next_word)\n    word_for_last = idx2word[id_for_last]\n    print(word_for_last)\n\n    return prob_correct\n\n\n\n\npredict_next(model, y_train[0], X_train[0], word2idx)\n\n#Function to calculate perplexity\ndef calculate_perplexity(words):\n    # PP = (P (tn | t1,..., t_n-1) ... P(t_3|t_1, t_2)P(t_2|t_1)P(t_1))^{-1/n}\n    n = len(words)\n    p_insid

In [31]:
unknown_idx = word2idx.get('<unk>', None)
X_train_tensor = torch.tensor([[word2idx.get(w, unknown_idx) for w in sentence] for sentence in X_train]).to(device)
y_train_tensor = torch.tensor([word2idx.get(w, unknown_idx) for w in y_train]).to(device)

X_valid_tensor = torch.tensor([[word2idx.get(w, unknown_idx) for w in sentence] for sentence in X_valid]).to(device)
y_valid_tensor = torch.tensor([word2idx.get(w, unknown_idx) for w in y_valid]).to(device)

num_epochs = 10

#Training using a smoothed trigram
best_val_perplexity = np.inf
epochs_without_improvement = 0
patience = 3 
batch_size = 64

for epoch in range(num_epochs):
    model.train() 
    total_loss = 0
    num_batch = 0
    for batch_idx in range(0, len(X_train_tensor), batch_size):
        X_batch = X_train_tensor[batch_idx:batch_idx+batch_size]
        y_batch = y_train_tensor[batch_idx:batch_idx+batch_size]

        h0 = torch.zeros(num_layers, X_batch.size(0), hidden_size, device=device)
        c0 = torch.zeros(num_layers, X_batch.size(0), hidden_size, device=device)

        pred, _ = model(X_batch, (h0,c0))
        #pred_reshaped = pred.reshape(X_batch.size(0), X_batch.size(1), -1)
        pred_reshaped = pred.reshape(X_batch.size(0), X_batch.size(1), -1)
        pred_last = pred_reshaped[:, -1, :]
        #pred_flat = pred.reshape(-1, vocab_size)
        #print(pred_flat.shape)
        y_flattened = y_batch.reshape(-1)
        #print(y_flattened.shape)
        loss = criterion(pred_last, y_flattened)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batch += 1
    model.eval()
    val_total_loss = 0
    val_num_batch = 0
    with torch.no_grad():
        for val_batch_idx in range(0, len(X_valid_tensor), batch_size):
            x_val_batch = X_valid_tensor[val_batch_idx:val_batch_idx+batch_size]
            y_val_batch = y_valid_tensor[val_batch_idx: val_batch_idx+ batch_size]
        
            h0_val = torch.zeros(num_layers, x_val_batch.size(0), hidden_size, device=device)
            c0_val = torch.zeros(num_layers, x_val_batch.size(0), hidden_size, device=device)

            predicted_val, _ = model(x_val_batch, (h0_val, c0_val))

            predicted_val_reshaped = predicted_val.reshape(x_val_batch.size(0), x_val_batch.size(1), -1)
#y_valid_flat = predicted_val.reshape(x_val_batch.size(0), x_val_batch.size(1), -1)
            pred_last_val = predicted_val_reshaped[:, -1, :]

            y_valid_flat = y_val_batch.view(-1)
            
        
            #_, pred_val_val = torch.max(predicted_val, 1)
            val_loss = criterion(pred_last_val, y_valid_flat)
            val_total_loss += val_loss.item()
            val_num_batch += 1             
    avg_train_loss = total_loss / num_batch
    avg_val_loss = val_total_loss / val_num_batch
    val_perplexity = torch.exp(torch.tensor(avg_val_loss)).item()
    print(f"Epoch {epoch}, loss = {avg_train_loss:.4f}, val_perplexity = {val_perplexity:.2f}")
    if val_perplexity < best_val_perplexity:
        best_val_perplexity = val_perplexity
        epochs_without_improvement = 0
        print("New best")
    else:
        epochs_without_improvement +=1
    if epochs_without_improvement >= patience:
        print("No improvement")
        break


print(f"best is {best_val_perplexity}")

Epoch 0, loss = 5.7022, val_perplexity = 194.73
New best
Epoch 1, loss = 5.0479, val_perplexity = 161.55
New best
Epoch 2, loss = 4.7846, val_perplexity = 153.75
New best
Epoch 3, loss = 4.6234, val_perplexity = 153.14
New best
Epoch 4, loss = 4.5132, val_perplexity = 154.07
Epoch 5, loss = 4.4282, val_perplexity = 154.44
Epoch 6, loss = 4.3640, val_perplexity = 155.65
No improvement
best is 153.14328002929688


publishing


Implement a smoothed triagram model
Then optimise the two smoothing parameters by traial and error to get the lowest validation perplexity

In [ ]:
from collections import defaultdict, Counter
from scipy.optimize import minimize

unigram_counts = Counter(tokens)
bigram_counts = Counter([(tokens[i], tokens[i+1]) for i in range(len(tokens)-1) ])
trigram_counts = Counter([(tokens[i], tokens[i+1], tokens[i+2]) for i in range(len(tokens)-2) ])

def perp(w_cur, w_prev, w_prev_prev, c1, c2):
    prob_unigram = unigram_counts[w_cur] / sum(unigram_counts.values())
    prob_bigram = bigram_counts[(w_prev, w_cur)] / unigram_counts[w_prev] if unigram_counts[w_prev] > 0 else 0
    prob_trigram = trigram_counts[(w_prev_prev, w_prev, w_cur)] / bigram_counts[(w_prev_prev, w_prev)] if bigram_counts[(w_prev_prev, w_prev)] > 0 else 0
    return c1 * prob_trigram + c2 *prob_bigram + (1 - c1 - c2) * prob_unigram






def min_loss(params, training):

    c1, c2 = params[0], params[1]

    
    predicted = np.zeros_like(training)
    error = np.zeros_like(training)

    for t in range(max(p,q), len(training)):
        next = c
        for i in range(len(phi)):
            next += phi[i] * training[t - i - 1]
        for j in range(len(theta)):
            next += theta[j] * error[t - j - 1]
        predicted[t] = next
        error[t] = training[t] - predicted[t]
    
    n = len(training) - max(p, q)
    if sigma2 <= 0:
        return np.inf
    #var = np.mean(error[max(p, q):] ** 2)
    log_likelihood = - (n/2) * np.log(2 * np.pi) - (n/2) * np.log(sigma2) - (1/(2*sigma2)) * np.sum(error[max(p,q):]**2)
    return -log_likelihood


def arma_inference(training):

    initial_params = np.full(2, 0.1)
    res = minimize(min_loss, initial_params, args = (training), method='Nelder-Mead', tol=1e-4)

    return(res)



TypeError: predict_next() missing 1 required positional argument: 'c_vals'